# Lesson 7.1 - Agentic AI Foundations & Architectures (toy agent demo)

## Objectives

- Build a minimal plan-act-observe loop agent.
- Represent agent state explicitly.
- Connect architecture choices to reliability and business outcomes.
- Practice evaluating the difference between chatbot behavior and outcome-driven agent behavior.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List
import random

random.seed(42)


## Agent Environment

We simulate a lightweight "email triage and scheduling" environment.

- Input: inbound messages.
- Actions: classify, draft response, propose meeting slots, escalate.
- Objective: complete each message with policy-safe handling.


In [ ]:
EMAILS = [
    {
        "id": "E-100",
        "from": "client-a@example.com",
        "subject": "Need onboarding call",
        "body": "Can we schedule onboarding next week?",
        "priority": "medium",
    },
    {
        "id": "E-101",
        "from": "vip@example.com",
        "subject": "Contract issue",
        "body": "Billing amount looks wrong. Need urgent review.",
        "priority": "high",
    },
    {
        "id": "E-102",
        "from": "candidate@example.com",
        "subject": "General question",
        "body": "Do you provide API docs?",
        "priority": "low",
    },
]

SLOTS = ["Mon 10:00", "Tue 14:00", "Wed 16:30"]


## State and Planner

A key idea in agent engineering: make state explicit and inspectable.


In [ ]:
@dataclass
class AgentState:
    email: Dict[str, Any]
    status: str = "new"
    intent: str | None = None
    actions_taken: List[str] = field(default_factory=list)
    draft_response: str | None = None
    escalated: bool = False
    done: bool = False


def decide_next_action(state: AgentState) -> str:
    if state.intent is None:
        return "classify_intent"
    if state.intent == "meeting" and state.draft_response is None:
        return "propose_slots"
    if state.intent == "issue" and not state.escalated:
        return "escalate"
    if state.draft_response is None:
        return "draft_response"
    return "close"


## Action Functions

These functions act as tools the agent can call.


In [ ]:
def classify_intent_tool(state: AgentState) -> None:
    text = (state.email["subject"] + " " + state.email["body"]).lower()
    if "schedule" in text or "onboarding" in text or "call" in text:
        state.intent = "meeting"
    elif "issue" in text or "wrong" in text or "urgent" in text:
        state.intent = "issue"
    else:
        state.intent = "faq"
    state.actions_taken.append(f"intent={state.intent}")


def propose_slots_tool(state: AgentState) -> None:
    slot = random.choice(SLOTS)
    state.draft_response = (
        f"Thanks for reaching out. Available slot: {slot}. "
        "Please confirm if this works for you."
    )
    state.actions_taken.append("slot_proposed")


def escalate_tool(state: AgentState) -> None:
    state.escalated = True
    state.draft_response = (
        "I have escalated this to our billing specialist and marked it urgent. "
        "You will receive an update shortly."
    )
    state.actions_taken.append("escalated_to_human")


def draft_response_tool(state: AgentState) -> None:
    state.draft_response = "Thanks for your message. You can find API docs at /docs/api."
    state.actions_taken.append("faq_response_drafted")


def close_tool(state: AgentState) -> None:
    state.status = "closed"
    state.done = True
    state.actions_taken.append("closed")


## Plan-Act-Observe Loop

This loop is the structural difference between an agent and a single-turn completion.


In [ ]:
ACTION_MAP = {
    "classify_intent": classify_intent_tool,
    "propose_slots": propose_slots_tool,
    "escalate": escalate_tool,
    "draft_response": draft_response_tool,
    "close": close_tool,
}


def run_agent(email: Dict[str, Any], max_steps: int = 6) -> AgentState:
    state = AgentState(email=email)
    for _ in range(max_steps):
        action = decide_next_action(state)
        ACTION_MAP[action](state)
        if state.done:
            break
    return state


results = [run_agent(email) for email in EMAILS]
for st in results:
    print(st.email["id"], st.intent, st.escalated, st.status)
    print("  actions:", st.actions_taken)
    print("  draft:", st.draft_response)


## Optional Extension: LLM-assisted planning

If an LLM is available, replace `decide_next_action` with a policy function that predicts the next tool call from structured state. Keep hard guardrails in deterministic code.


## Connect to Theory

- `AgentState` represents internal world state.
- `decide_next_action` is the policy layer.
- Tool functions represent environment actions.
- The loop forms the plan-act-observe control structure.

In production, add:

- action-level audit logs,
- timeout/retry strategy,
- permission checks,
- human approval gates for sensitive actions.


## Business Case Studies & Exceptions

### Case Study A: Knowledge Worker Automation

An enterprise email operations team used an agentic triage workflow to classify inbound requests, draft first responses, and route high-risk issues to specialists. The gain came from reducing context switching, not replacing expert judgment.

### Case Study B: Customer Onboarding Operations

A planner-executor agent pair automated account setup and internal ticket creation. Throughput improved, but production value appeared only after adding strict policy checks and traceability.

### Exceptions

- Legal, financial, and compliance-heavy requests should require explicit human review.
- If request volume is very low, a simple rule engine can outperform a full agent stack on cost.


## Interview Questions & Answers

1. **What makes an AI system "agentic"?**  
   Goal-driven multi-step behavior with action selection, tool use, and feedback adaptation.
2. **Why is explicit state useful in agent systems?**  
   It improves debugging, testing, and reproducibility.
3. **What is the difference between policy and tool layers?**  
   Policy selects actions; tools execute actions.
4. **When do you introduce human-in-the-loop?**  
   For high-risk decisions or low-confidence outputs.
5. **How can a simple loop fail in production?**  
   Infinite loops, wrong routing, and unsafe tool calls.
6. **How do you evaluate this toy agent?**  
   Task success, escalation precision, latency, and action count.
7. **What is an escalation policy?**  
   A rule set deciding when automation stops and humans intervene.
8. **Why not rely on LLM decisions for all steps?**  
   Deterministic guardrails are more reliable for safety-critical controls.
9. **How does memory differ from logs?**  
   Memory guides future decisions; logs are immutable records for analysis.
10. **What is the first production hardening step?**  
   Add typed tool schemas and action-level observability.
